# FICOS (Freight Intelligence & Charter Optimization System)
## Phase 6: XGBoost Regression Forecasting & Benchmark Comparison

### 1. Objective
This notebook implements, evaluates, and diagnoses **XGBoost (Gradient Boosted Decision Trees)** for 1-step-ahead forecasting of Baltic dry-bulk freight sub-indices (`HSI`, `SI`, `PI`, `CI`) on the historical dataset.

**Core Benchmark Question:**
> Can a non-linear tree-based model improve upon the Phase 5 Ridge regression baseline?

### 2. Imports & Setup

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb

# Add project root to sys.path
repo_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.models.evaluation import (
    split_chronological_holdout,
    compute_regression_metrics,
    run_phase6_xgboost_experiment,
)
from src.models.xgboost_model import XGBoostForecaster
from src.data.schemas import DATE_COLUMN, TARGET_COLUMNS

print(f'XGBoost Version: {xgb.__version__}')
sns.set_theme(style='whitegrid', font_scale=1.0)
plt.rcParams['figure.figsize'] = (12, 6)

### 3. Load Feature Matrix & Chronological 80/20 Holdout Split

In [ ]:
feat_path = repo_root / 'data' / 'features' / 'freight_features.csv'
df_feat = pd.read_csv(feat_path)
df_feat[DATE_COLUMN] = pd.to_datetime(df_feat[DATE_COLUMN])

train_df, test_df = split_chronological_holdout(df_feat, train_ratio=0.80, drop_initial_cold_start=21)
print(f'Total Features Matrix: {df_feat.shape[0]:,} rows x {df_feat.shape[1]} columns')
print(f'Train Set: {len(train_df):,} sessions ({train_df["date"].min().date()} to {train_df["date"].max().date()})')
print(f'Test Set:  {len(test_df):,} sessions ({test_df["date"].min().date()} to {test_df["date"].max().date()})')

### 4. Execute Full Phase 6 XGBoost Benchmark Experiment

In [ ]:
config_path = repo_root / 'configs' / 'models.yaml'
metrics_df, pred_df, imp_df, meta = run_phase6_xgboost_experiment(config_path)
print('Experiment Completed Successfully.')

### 5. Performance Comparison: Persistence vs Ridge vs XGBoost

In [ ]:
display(metrics_df) if 'display' in globals() else print(metrics_df.to_string(index=False))

### 6. Summary Comparison by Metric (MAE & Directional Accuracy)

In [ ]:
pivot_mae = metrics_df.pivot(index='model', columns='target', values='mae')
print('MAE Across Models:')
display(pivot_mae) if 'display' in globals() else print(pivot_mae)

pivot_da = metrics_df.pivot(index='model', columns='target', values='da_pct')
print('\nDirectional Accuracy (%) Across Models:')
display(pivot_da) if 'display' in globals() else print(pivot_da)

### 7. Feature Importance Analysis (Tree Gain Metric)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()
for i, tgt in enumerate(['bdi_hsi', 'bdi_si', 'bdi_pi', 'bdi_ci']):
    sub = imp_df[imp_df['target'] == tgt].head(8).sort_values(by='importance', ascending=True)
    axes[i].barh(sub['feature'], sub['importance'], color='#2ca02c')
    axes[i].set_title(f'Top 8 Features for {tgt.upper()} (Gain)', fontweight='bold')
    axes[i].set_xlabel('Gain Score')

plt.tight_layout()
plt.show()

### 8. Forecasts vs Ground Truth in Test Period

In [ ]:
dates = pd.to_datetime(pred_df['date'])
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()
for i, tgt in enumerate(['bdi_hsi', 'bdi_si', 'bdi_pi', 'bdi_ci']):
    axes[i].plot(dates, pred_df[f'actual_{tgt}'], label='Actual Ground Truth', color='black', lw=1.5)
    axes[i].plot(dates, pred_df[f'pred_{tgt}_ridge'], label='Ridge', linestyle='--', color='#1f77b4', alpha=0.8)
    axes[i].plot(dates, pred_df[f'pred_{tgt}_xgboost'], label='XGBoost', color='#2ca02c', lw=1.3)
    axes[i].set_title(f'{tgt.upper()} Test Period Forecasts', fontweight='bold')
    axes[i].set_ylabel('Index Level')
    axes[i].legend(loc='upper left')

plt.tight_layout()
plt.show()

### 9. Residual Error Diagnostics & Capesize Non-Linear Behavior

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()
for i, tgt in enumerate(['bdi_hsi', 'bdi_si', 'bdi_pi', 'bdi_ci']):
    actual = pred_df[f'actual_{tgt}']
    xgb_p = pred_df[f'pred_{tgt}_xgboost']
    res = actual - xgb_p
    sns.histplot(res, kde=True, ax=axes[i], color='#2ca02c', bins=30)
    axes[i].set_title(f'{tgt.upper()} XGBoost Residual Distribution (std={res.std():.2f})', fontweight='bold')
    axes[i].set_xlabel('Error')

plt.tight_layout()
plt.show()

### 10. Key Findings & Phase 7 Strategy
1. **Ridge Outperforms Default XGBoost on Raw Index Levels**: Linear models extrapolate continuous autoregressive slopes smoothly, whereas decision trees partition continuous levels into step-wise constant buckets.
2. **Dominance of Autoregressive & Cross-Vessel Signals**: Over 80% of tree gain originates from lag-1 levels and cross-segment spillover features.
3. **Recommendation for Next Phase**: Formulate targets as returns/differences ($\Delta Y_{t+1}$) or train hybrid Ridge + residual boosting models.